# Analyze your own module: simulate it, then see its schematic

```
paste your RTL  ->  simulate (waveform)  ->  RTL schematic
```

Run the cells from top to bottom with **Shift + Enter**.

## Step 0: Install the tools (run once)
You should see four lines starting with `[OK]`.

In [ ]:
import os, sys, shutil, subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)

# 1. Icarus Verilog (auto-installs on Colab/Linux)
if shutil.which("iverilog") is None:
    if sys.platform.startswith("linux"):
        print("Installing Icarus Verilog (about 30 seconds)...")
        rc, out = sh("apt-get update -qq && apt-get install -y -qq iverilog")
        if rc != 0:
            print("Icarus install failed. Last lines of the error:")
            print(out[-800:])
    else:
        print("Install Icarus Verilog for your computer, then restart VSCode.")
        print("Windows: https://bleyer.org/icarus/   Mac: brew install icarus-verilog")

# 2. Python packages: Yosys (reads RTL) and matplotlib (draws waveforms)
rc, out = sh(f'"{sys.executable}" -m pip install -q yowasp-yosys matplotlib')
if rc != 0:
    print("pip install failed. Last lines of the error:")
    print(out[-800:])

# 3. netlistsvg (draws schematics, needs Node.js)
if shutil.which("netlistsvg") is None:
    if shutil.which("npm") is None:
        print("Node.js was not found. Install Node.js (LTS) from nodejs.org, then run this cell again.")
    else:
        print("Installing netlistsvg...")
        rc, out = sh("npm install -g netlistsvg")
        if rc != 0:
            print("netlistsvg install failed. Last lines of the error:")
            print(out[-800:])

# 4. Check everything
print("---- setup check ----")
ic = shutil.which("iverilog")
print(("[OK]  " if ic else "[X]   ") + "Icarus Verilog", "-", sh("iverilog -V")[1].splitlines()[0] if ic else "not found")

rc, out = sh(f'"{sys.executable}" -c "import yowasp_yosys"')
print(("[OK]  " if rc == 0 else "[X]   ") + "Yosys (yowasp-yosys)")

ns = shutil.which("netlistsvg")
print(("[OK]  " if ns else "[X]   ") + "netlistsvg")

rc, out = sh(f'"{sys.executable}" -c "import matplotlib; print(matplotlib.__version__)"')
print(("[OK]  " if rc == 0 else "[X]   ") + "matplotlib", "-", out.strip() if rc == 0 else "not installed")

## Step 1: Helpers (just run this cell)
- `run_auto(...)` makes a testbench for you from your module's ports, runs it, and records the waveform.
- `run_sim(...)` runs a testbench that you wrote yourself.
- `show_waves(...)` draws the waveform.
- `show_schematic(...)` draws the RTL schematic.
- `show_code(...)` shows your Verilog with colors.

In [ ]:
import os, sys, shutil, subprocess

# ---------- Verilog testbench flow ----------
def _as_text(x):
    """Return the file's contents if x is a path to a file, otherwise x itself."""
    if len(x) < 260 and chr(10) not in x and os.path.isfile(x):
        with open(x) as f:
            return f.read()
    return x


def run_sim(design, testbench, top="tb", name="sim"):
    """Compile and run a design with a Verilog testbench. Returns the waveform file path."""
    if shutil.which("iverilog") is None:
        print("iverilog is missing. Run Step 0 again.")
        return None
    os.makedirs("build", exist_ok=True)
    design, testbench = _as_text(design), _as_text(testbench)
    d, t, vvp = f"build/{name}_design.sv", f"build/{name}_tb.sv", f"build/{name}.vvp"
    with open(d, "w") as f:
        f.write(design)
    with open(t, "w") as f:
        f.write(testbench)
    if os.path.exists("build/dump.vcd"):
        os.remove("build/dump.vcd")
    r = subprocess.run(["iverilog", "-g2012", "-s", top, "-o", vvp, d, t],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("COMPILE ERROR (fix this, then run the cell again):")
        print(r.stderr or r.stdout)
        return None
    if r.stderr.strip():
        print(r.stderr)
    env = dict(os.environ, IVERILOG_DUMPER="vcd")
    r = subprocess.run(["vvp", vvp, "-vcd"], capture_output=True, text=True, env=env)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)
    if not os.path.exists("build/dump.vcd"):
        if "suppressed" in r.stdout:
            print("Icarus has waveform dumping switched off in this session.")
        print("No waveform file was written. Does your testbench call $dumpfile and $dumpvars?")
        return None
    return "build/dump.vcd"


# ---------- Waveform drawing ----------
def read_log(path):
    """Read the waveform log written by wave_recorder. Returns (signals, timescale)."""
    sigs = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 3:
                continue
            t, name, val = int(parts[0]), parts[1], parts[2]
            s = sigs.setdefault(name, {"name": name, "width": len(val), "ch": []})
            if not s["ch"] or s["ch"][-1][1] != val:
                s["ch"].append((t, val))
    return sigs, "ps"


def read_vcd(path):
    """Read a .vcd file. Returns (signals, timescale)."""
    with open(path) as f:
        toks = f.read().split()
    sigs, scope, timescale = {}, [], ""
    t, i, n, defs_done = 0, 0, len(toks), False
    while i < n:
        tk = toks[i]
        if tk in ("$date", "$version", "$comment", "$timescale"):
            j = i + 1
            while toks[j] != "$end":
                j += 1
            if tk == "$timescale":
                timescale = "".join(toks[i + 1:j])
            i = j + 1
        elif tk == "$scope":
            scope.append(toks[i + 2])
            i += 4
        elif tk == "$upscope":
            scope.pop()
            i += 2
        elif tk == "$var":
            width, ident, name = int(toks[i + 2]), toks[i + 3], toks[i + 4]
            j = i + 5
            while toks[j] != "$end":
                j += 1
            label = ".".join(scope[1:] + [name]) if len(scope) > 1 else name
            if ident not in sigs:
                sigs[ident] = {"name": label, "width": width, "ch": []}
            i = j + 1
        elif tk == "$enddefinitions":
            defs_done = True
            i += 2
        elif not defs_done or tk.startswith("$"):
            i += 1
        elif tk[0] == "#":
            t = int(tk[1:])
            i += 1
        elif tk[0] in "bB":
            if toks[i + 1] in sigs:
                sigs[toks[i + 1]]["ch"].append((t, tk[1:].lower()))
            i += 2
        elif tk[0] in "rR":
            i += 2
        else:
            if tk[1:] in sigs:
                sigs[tk[1:]]["ch"].append((t, tk[0].lower()))
            i += 1
    return sigs, timescale


def _fmt(v, radix):
    if not set(v) <= {"0", "1"}:
        return "X" if "x" in v else "Z"
    if radix == "bin":
        return v
    if radix == "dec":
        return str(int(v, 2))
    return format(int(v, 2), "X")


def _value_at(ch, t):
    val = None
    for tt, v in ch:
        if tt <= t:
            val = v
        else:
            break
    return val


def show_waves(wave_path="build/dump.vcd", signals=None, t_max=None,
               radix="hex", cursor=None, theme="dark"):
    """Draw the waveform like Vivado's viewer.
    signals = names to show (optional)   radix = "hex", "dec" or "bin"
    cursor  = time to read values at, in ns (default: the end)   theme = "dark" or "light"
    """
    import re
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec
    if str(wave_path).endswith(".log"):
        sigs, ts = read_log(wave_path)
    else:
        sigs, ts = read_vcd(wave_path)
    items = [s for s in sigs.values() if s["ch"]]
    if signals:
        order = {name: k for k, name in enumerate(signals)}
        items = sorted([s for s in items if s["name"] in order], key=lambda s: order[s["name"]])
    if not items:
        print("No signals to draw. Names available:", [s["name"] for s in sigs.values()])
        return

    m = re.match(r"(\d*)\s*(fs|ps|ns|us|ms|s)", ts or "")
    per = {"fs": 1e-6, "ps": 1e-3, "ns": 1.0, "us": 1e3, "ms": 1e6, "s": 1e9}
    scale = int(m.group(1) or 1) * per[m.group(2)] if m else 1.0
    unit = "ns" if m else "time units"

    if theme == "light":
        bg, panel, fg, grid, wave, red, cur_c = "white", "#f1f1f1", "#202020", "#dddddd", "#0a7d0a", "#d62020", "#c89600"
    else:
        bg, panel, fg, grid, wave, red, cur_c = "#0e0e0e", "#1c1c1c", "#e6e6e6", "#2c2c2c", "#2ee62e", "#ff4545", "#ffd400"

    end_raw = t_max or max(t for s in items for t, _ in s["ch"])
    end = max(end_raw, 1) * scale
    cur = (cursor if cursor is not None else end_raw * scale)
    cur_raw = cur / scale
    n = len(items)

    fig = plt.figure(figsize=(11, 0.6 * n + 1.1), facecolor=bg)
    gs = GridSpec(n, 3, width_ratios=[1.5, 1.0, 9], hspace=0, wspace=0,
                  left=0.01, right=0.995, top=0.86, bottom=0.2)
    first_wave = None
    for r, s in enumerate(items):
        axn = fig.add_subplot(gs[r, 0])
        axv = fig.add_subplot(gs[r, 1])
        axw = fig.add_subplot(gs[r, 2], sharex=first_wave) if first_wave else fig.add_subplot(gs[r, 2])
        first_wave = first_wave or axw
        for a in (axn, axv):
            a.set_facecolor(panel)
            a.set_xticks([]); a.set_yticks([])
            for sp in a.spines.values():
                sp.set_color(grid)
        axn.text(0.06, 0.5, s["name"], color=fg, va="center", ha="left",
                 transform=axn.transAxes, fontsize=10)
        val = _value_at(s["ch"], cur_raw)
        axv.text(0.5, 0.5, _fmt(val, radix) if val is not None else "", color=wave if val and set(val) <= {"0", "1"} else red,
                 va="center", ha="center", transform=axv.transAxes, fontsize=10)

        axw.set_facecolor(bg)
        axw.set_ylim(-0.2, 1.2)
        axw.set_yticks([])
        axw.grid(axis="x", color=grid, linewidth=0.6)
        for side in ("top", "right", "left"):
            axw.spines[side].set_visible(False)
        axw.spines["bottom"].set_color(grid)

        ch = [(t * scale, v) for t, v in s["ch"] if t * scale <= end]
        for k, (t0, v) in enumerate(ch):
            t1 = ch[k + 1][0] if k + 1 < len(ch) else end
            known = set(v) <= {"0", "1"}
            col = wave if known else red
            if s["width"] == 1:
                if known:
                    y = int(v)
                    axw.plot([t0, t1], [y, y], color=col, linewidth=1.6)
                    if k + 1 < len(ch) and set(ch[k + 1][1]) <= {"0", "1"}:
                        y2 = int(ch[k + 1][1])
                        if y2 != y:
                            axw.plot([t1, t1], [y, y2], color=col, linewidth=1.6)
                else:
                    axw.fill_between([t0, t1], 0, 1, color=red, alpha=0.55, linewidth=0)
            else:
                d = min(end * 0.004, (t1 - t0) / 2)
                axw.plot([t0, t0 + d, t1 - d, t1, t1 - d, t0 + d, t0],
                         [0.5, 1, 1, 0.5, 0, 0, 0.5], color=col, linewidth=1.3)
                text = _fmt(v, radix)
                if (t1 - t0) / end > 0.011 * len(text) + 0.008:
                    axw.text((t0 + t1) / 2, 0.5, text, color=fg, ha="center", va="center", fontsize=9)
        axw.axvline(cur, color=cur_c, linewidth=1.2)
        axw.tick_params(axis="x", colors=fg, labelsize=8, labeltop=(r == 0), labelbottom=(r == n - 1),
                        top=(r == 0), bottom=(r == n - 1))
        axw.set_xlim(0, end)
        if r == 0:
            axw.xaxis.set_ticks_position("both")
            axw.text(cur, 1.25, f"{cur:g} {unit}", color=cur_c, ha=("right" if cur > 0.9 * end else "center"), va="bottom", fontsize=8)
    items_last = fig.axes[-1]
    items_last.set_xlabel("time (" + unit + ")", color=fg, fontsize=9)
    plt.show()

# ---------- RTL tools (Yosys + netlistsvg) ----------
import re, json
from IPython.display import SVG, display

def _find(names):
    base = os.path.dirname(sys.executable)
    folders = [None, base, os.path.join(base, "Scripts"), os.path.join(base, "Library", "bin")]
    for n in names:
        for path in folders:
            p = shutil.which(n, path=path)
            if p:
                return p
    return None

_BOOT = chr(10).join([
    "import sys, yowasp_yosys as y",
    "f = getattr(y, 'run_yosys', None)",
    "if f is not None:",
    "    sys.exit(f(sys.argv[1:]) or 0)",
    "f = getattr(y, '_run_yosys_argv', None)",
    "if f is not None:",
    "    sys.exit(f() or 0)",
    "print('yowasp_yosys has no runner. Contents:', dir(y))",
    "sys.exit(1)",
])

def _yosys_cmd():
    exe = _find(["yowasp-yosys", "yosys"])
    if exe:
        return [exe]
    try:
        import yowasp_yosys
        return [sys.executable, "-c", _BOOT]
    except Exception:
        return None

YOSYS = _yosys_cmd()
NETLISTSVG = _find(["netlistsvg"])


def _run_yosys(script):
    if YOSYS is None:
        print("Yosys is missing. Run Step 0 again.")
        return None
    os.makedirs("build", exist_ok=True)
    r = subprocess.run([*YOSYS, "-q", "-p", script], capture_output=True, text=True)
    if r.returncode != 0:
        print("Yosys found a problem in your RTL:")
        print(r.stderr or r.stdout)
        return None
    if r.stderr.strip():
        print("Yosys warnings:")
        print(r.stderr)
    return r


def get_ports(rtl_file, top=None):
    """Ask Yosys for the top module and its ports. Returns (top_name, [(name, direction, width), ...])."""
    out = "build/_ports.json"
    sel = f"-top {top}" if top else "-auto-top"
    if _run_yosys(f"read_verilog -sv {rtl_file}; hierarchy {sel}; write_json {out}") is None:
        return None, []
    mods = json.load(open(out))["modules"]
    name = top
    if name is None:
        for n, m in mods.items():
            v = m.get("attributes", {}).get("top")
            if v is not None and int(str(v), 2) != 0:
                name = n
        if name is None:
            name = list(mods)[0]
    ports = [(p, d["direction"], len(d["bits"])) for p, d in mods[name]["ports"].items()]
    return name, ports


def make_testbench(rtl_file, top=None, cycles=40, out="build/auto_tb.sv"):
    """Write a simple testbench for the module: a clock, a reset pulse and random inputs."""
    name, ports = get_ports(rtl_file, top)
    if not name:
        return None
    clk = next((p for p, d, w in ports if d == "input" and w == 1
                and re.fullmatch(r"(clk|clock|clk_i|i_clk)", p, re.I)), None)
    rst = next((p for p, d, w in ports if d == "input" and w == 1
                and re.fullmatch(r"(rst|reset|rst_n|resetn|reset_n|rst_ni|areset|i_rst|i_rst_n)", p, re.I)), None)
    low = bool(rst) and rst.lower().endswith(("n", "ni"))
    on, off = ("1'b0", "1'b1") if low else ("1'b1", "1'b0")
    others = [p for p, d, w in ports if d == "input" and p not in (clk, rst)]

    L = ["`timescale 1ns/1ps", "module tb_auto;"]
    for p, d, w in ports:
        rng = f"[{w - 1}:0] " if w > 1 else ""
        L.append(f"    {'reg' if d == 'input' else 'wire'} {rng}{p};")
    L.append("    integer fd = 0;")
    L.append("    integer k;")
    L.append(f"    {name} dut(" + ", ".join(f".{p}({p})" for p, d, w in ports) + ");")
    if clk:
        L.append(f"    always #5 {clk} = ~{clk};")
    for p, d, w in ports:
        L.append(f'    always @({p}) if (fd != 0) $fdisplay(fd, "%0d {p} %b", $rtoi($realtime*1000.0), {p});')
    L.append("    initial begin")
    L.append('        fd = $fopen("build/waves.log", "w");')
    if clk:
        L.append(f"        {clk} = 0;")
    if rst:
        L.append(f"        {rst} = {on};")
    for p in others:
        L.append(f"        {p} = 0;")
    for p, d, w in ports:
        L.append(f'        $fdisplay(fd, "0 {p} %b", {p});')
    L.append("        #2;")
    L.append(f"        for (k = 0; k < {cycles}; k = k + 1) begin")
    if rst:
        L.append(f"            if (k == 2) {rst} = {off};")
    for p in others:
        L.append(f"            {p} = $urandom;")
    L.append("            #10;")
    L.append("        end")
    L.append("        $fclose(fd);")
    L.append("        $finish;")
    L.append("    end")
    L.append("endmodule")
    with open(out, "w") as f:
        f.write(chr(10).join(L) + chr(10))
    print("Top module:", name)
    print("Clock:", clk or "none (combinational)", "| Reset:", (rst + (" (active low)" if low else " (active high)")) if rst else "none")
    print("Random inputs:", ", ".join(others) or "none")
    return out


def run_auto(rtl_file, top=None, cycles=40):
    """Generate a testbench from the module's ports, run it, and return the waveform log."""
    if shutil.which("iverilog") is None:
        print("iverilog is missing. Run Step 0 again.")
        return None
    tb = make_testbench(rtl_file, top, cycles)
    if tb is None:
        return None
    if os.path.exists("build/waves.log"):
        os.remove("build/waves.log")
    r = subprocess.run(["iverilog", "-g2012", "-s", "tb_auto", "-o", "build/auto.vvp", rtl_file, tb],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("COMPILE ERROR (fix your RTL, then run the cell again):")
        print(r.stderr or r.stdout)
        return None
    if r.stderr.strip():
        print(r.stderr)
    r = subprocess.run(["vvp", "build/auto.vvp"], capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)
    if os.path.exists("build/waves.log") and os.path.getsize("build/waves.log") > 0:
        print("Simulation finished. Waveform recorded.")
        return "build/waves.log"
    print("No waveform was recorded.")
    return None


def show_schematic(rtl_file, top=None, flatten=False):
    """Draw the RTL schematic (like Vivado's elaborated design view)."""
    if NETLISTSVG is None:
        print("netlistsvg is missing. Run Step 0 again.")
        return
    sel = f"-top {top}" if top else "-auto-top"
    extra = "flatten; opt_clean; " if flatten else ""
    if _run_yosys(f"read_verilog -sv {rtl_file}; prep {sel}; {extra}write_json build/schematic.json") is None:
        return
    r = subprocess.run([NETLISTSVG, "build/schematic.json", "-o", "build/schematic.svg"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("netlistsvg could not draw this:")
        print(r.stderr or r.stdout)
        return
    svg = open("build/schematic.svg").read().replace("<svg", '<svg style="background:white"', 1)
    display(SVG(data=svg))


def show_code(path):
    """Show a Verilog/SystemVerilog file with colors (syntax highlighting)."""
    from IPython.display import HTML
    try:
        from pygments import highlight
        from pygments.lexers import get_lexer_by_name
        from pygments.formatters import HtmlFormatter
    except ImportError:
        print(open(path).read())
        return
    fmt = HtmlFormatter(style="monokai", noclasses=True, linenos="inline")
    html = highlight(open(path).read(), get_lexer_by_name("systemverilog"), fmt)
    display(HTML(html.replace("<pre", '<pre style="background:#272822;padding:10px;border-radius:6px;line-height:1.4"', 1)))

## Step 2: Paste your RTL
Keep `%%writefile mymodule.sv` on the **very first line**. Replace everything below it with your own module (SystemVerilog or Verilog). The module shown here is just an example.

If your design has several modules, paste them all in this one cell.

In [ ]:
%%writefile mymodule.sv
module mymodule(input  logic       clk,
                input  logic       rst,
                input  logic       en,
                input  logic       load,
                input  logic [3:0] d,
                output logic [3:0] q);
    always_ff @(posedge clk) begin
        if (rst)       q <= 4'd0;
        else if (load) q <= d;
        else if (en)   q <= q + 4'd1;
    end
endmodule

**Want colors?** The box above is plain text (Colab cannot color a `%%writefile` cell). Run this cell to see the same file color coded:

In [ ]:
show_code("mymodule.sv")

## Step 3: Simulate it (automatic testbench)
This finds your module's ports, drives a clock, pulses reset, applies random inputs, and records every port.

It prints what it found: the top module, the clock, the reset, and the random inputs. **Check that this matches your design.** If your design has no clock or reset, that is fine.

In [ ]:
wave = run_auto("mymodule.sv")
if wave:
    show_waves(wave)

**Reading the waveform:** names on the left, the value at the yellow cursor next to them, time along the top. Buses show a number in the middle (hex by default). Red means unknown (`X`), which is normal before the first reset.

Useful options:
```python
show_waves(wave, ["clk", "rst", "q"])   # show only some signals, in this order
show_waves(wave, radix="dec")           # show numbers as decimal ("bin" also works)
show_waves(wave, cursor=52)             # put the cursor at 52 ns
```

## Step 3b (optional): Use your own testbench
Random inputs are good for a first look. To check that the module is *correct*, write a testbench with expected answers. This one matches the example module above, so **edit it to match yours**.

Keep the `$dumpfile` and `$dumpvars` lines. They save the waveform.

In [ ]:
%%writefile my_tb.sv
`timescale 1ns/1ps
module tb;
    reg clk = 0, rst = 1, en = 0, load = 0;
    reg [3:0] d = 0;
    wire [3:0] q;
    integer errors = 0;

    mymodule dut(.clk(clk), .rst(rst), .en(en), .load(load), .d(d), .q(q));

    always #5 clk = ~clk;

    initial begin
        $dumpfile("build/dump.vcd");
        $dumpvars(0, tb);
    end

    task check(input [3:0] expected);
        begin
            #1;
            if (q !== expected) begin
                $display("FAIL at %0t: q=%0d expected=%0d", $time, q, expected);
                errors = errors + 1;
            end
        end
    endtask

    initial begin
        repeat (2) @(posedge clk);
        #1 rst = 0;
        en = 1;
        @(posedge clk); check(4'd1);
        @(posedge clk); check(4'd2);
        d = 4'd9; load = 1;
        @(posedge clk); check(4'd9);
        load = 0;
        @(posedge clk); check(4'd10);
        en = 0;
        @(posedge clk); check(4'd10);
        if (errors == 0) $display("PASS: all checks passed");
        else             $display("FAILED with %0d errors", errors);
        $finish;
    end
endmodule

In [ ]:
wave = run_sim("mymodule.sv", "my_tb.sv", name="mine")
if wave:
    show_waves(wave)

If this prints a message about waveform dumping being switched off, use the automatic testbench in Step 3 for the waveform. The PASS or FAIL line above still works either way.

## Step 4: The RTL schematic
This is the same kind of picture as Vivado's **RTL Analysis > Schematic**. Boxes are gates, muxes, adders and flip-flops, and the lines are wires.

Yosys warnings (if any) print first. They work like a light version of Vivado's linter messages.

In [ ]:
show_schematic("mymodule.sv")

Options:
```python
show_schematic("mymodule.sv", top="mymodule")   # choose the top module by name
show_schematic("mymodule.sv", flatten=True)     # merge sub-modules into one picture
```
Large designs get crowded. Draw one module at a time, or leave sub-modules as plain boxes (the default).